# Eco-Logistics GRPO Training — v3 (hackathon-ready)

**Theme:** #3.1 World Modeling / Professional Tasks (primary), #2 Long-Horizon Planning (secondary).

**This version includes all bug fixes discovered during pre-onsite testing:**
- Pinned library versions (avoids dependency conflicts)
- `torchcodec` uninstall (fixes Unsloth import on Colab)
- `eager_generate` without mode-switching (fixes gradient corruption)
- `env.step()` returns 4-tuple not 3-tuple (matches actual env)
- `max_completion_length=128` (was 512 — wastes compute and learning signal)
- Auto-save + auto-download cell (survives Colab disconnects)

**Hardware guide:**
- Free T4 Colab: works but slow (~45 min for 100 steps with fp16)
- A100 / L4 (HF credits): fast (~15 min), switch `fp16=True` to `bf16=True`

**First-time setup:**
1. Set `HF_TOKEN` in Colab Secrets (left sidebar → 🔑 → Add new secret)
2. Update `ENV_URL` in cell 5 to your deployed Space
3. `Runtime → Change runtime type → T4 GPU` (or A100 if you have credits)
4. `Runtime → Run all`

**If training completes successfully, the last cell auto-downloads:**
- `training_curves_<TASK_ID>.png` — for the pitch deck
- `training_log_<TASK_ID>.csv` — raw numbers for the mini-blog
- `trajectory_before_<TASK_ID>.txt` — baseline agent's actions
- `trajectory_after_<TASK_ID>.txt` — trained agent's actions

## 1. Setup

In [ ]:
# All installs in ONE command so pip resolves dependencies together (avoids version conflicts)
!pip install -q "transformers>=4.56.0,<5.0" "trl[vllm]==0.24.0" openenv-core unsloth "pandas<3.0.0" "numpy<2.4" matplotlib "huggingface-hub>=0.34.0,<1.0"
# Uninstall torchcodec — it breaks Unsloth imports and we don't need video decoding
!pip uninstall -y torchcodec

**After cell 2 finishes, `Runtime → Restart session` then skip cell 2 on subsequent runs.**

In [ ]:
# HF login — reads from Colab Secrets (preferred) or environment variable
import os
from huggingface_hub import login

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, Exception):
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF login OK")
else:
    print("WARNING: HF_TOKEN not set — fine for training, needed only to push the LoRA at the end")

## 2. Configuration

In [ ]:
# ═══════ EDIT THESE BEFORE RUNNING ═══════
ENV_URL = os.environ.get("ENV_URL", "https://lokeshrao226-eco-logistics.hf.space")
TASK_ID = "inventory_balanced"   # options: restock_only, inventory_balanced, net_zero_profit
# ══════════════════════════════════════════

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# GRPO hyperparameters
NUM_GENERATIONS = 4
PER_DEVICE_BATCH = 1
GRAD_ACCUM_STEPS = 4
MAX_STEPS = 100
MAX_COMPLETION_LENGTH = 128   # actions are short JSON — 128 tokens is plenty, was 512
MAX_ENV_STEPS = 10
LEARNING_RATE = 5e-6

# World-modeling wrapper knobs (client-side, no env changes needed)
DEMAND_SHOCK_PROB = 0.15
DEMAND_SHOCK_MULT = 2.5
COMPETITOR_BID_PROB = 0.20
COMPETITOR_BID_MULT = 1.8

# A100/L4 users: change fp16=True to bf16=True in cell 23 for better stability
USE_BF16 = False   # Set True if on A100/L4 with compute capability 8.0+

REQUIRED_CONCURRENT = NUM_GENERATIONS * PER_DEVICE_BATCH * GRAD_ACCUM_STEPS
print(f"Training {MODEL_NAME} on task={TASK_ID}")
print(f"Env: {ENV_URL}")
print(f"HF Space must support max_concurrent_envs >= {REQUIRED_CONCURRENT}")

## 3. Smoke-test the env

In [ ]:
import requests
import uuid

# Reset with session ID (required for parallel-safe main.py v2.0)
session_id = f"smoketest-{uuid.uuid4().hex[:10]}"
headers = {"X-Session-Id": session_id}

r = requests.post(f"{ENV_URL}/reset", json={"task_id": TASK_ID}, headers=headers, timeout=30)
r.raise_for_status()
initial_obs = r.json()
print("Reset OK. Observation keys:", list(initial_obs.keys()))

dummy_action = {
    "ship_amount": 0.0,
    "origin_city": "Seattle",
    "destination_city": "Chicago",
    "speed_mode": "Rail",
}
r = requests.post(f"{ENV_URL}/step", json=dummy_action, headers=headers, timeout=30)
r.raise_for_status()
step_resp = r.json()
print("Step OK. Reward breakdown:", step_resp.get("reward"))
print("If you see both lines above, the env is good to train against.")

## 4. World-modeling wrapper (demand shocks + competitor)

In [ ]:
import random

CITIES = ["Seattle", "Chicago", "NYC"]

def apply_world_wrapper(obs: dict, rng: random.Random):
    modified = dict(obs)
    shock_state = {"demand_shocked": False, "contested_route": None}
    if rng.random() < DEMAND_SHOCK_PROB:
        shocked_demand = {c: v * DEMAND_SHOCK_MULT for c, v in obs["current_demand"].items()}
        modified["current_demand"] = shocked_demand
        shock_state["demand_shocked"] = True
    if rng.random() < COMPETITOR_BID_PROB:
        origin = rng.choice(CITIES)
        dest = rng.choice([c for c in CITIES if c != origin])
        shock_state["contested_route"] = (origin, dest)
        existing = modified.get("weather_alert") or ""
        note = f"Competitor bid up {origin}\u2192{dest} route ({COMPETITOR_BID_MULT}x cost this step)"
        modified["weather_alert"] = f"{existing}; {note}" if existing else note
    return modified, shock_state


def apply_reward_wrapper(reward_dict: dict, action: dict, shock_state: dict) -> dict:
    adjusted = dict(reward_dict)
    contested = shock_state.get("contested_route")
    if contested and action.get("ship_amount", 0) > 0:
        if (action.get("origin_city"), action.get("destination_city")) == contested:
            extra_cost = adjusted.get("shipping_cost", 0) * (COMPETITOR_BID_MULT - 1.0)
            adjusted["shipping_cost"] = adjusted.get("shipping_cost", 0) + extra_cost
    adjusted["total"] = (
        adjusted.get("sales_revenue", 0)
        - adjusted.get("shipping_cost", 0)
        - adjusted.get("carbon_penalty", 0)
        - adjusted.get("storage_fee", 0)
        + adjusted.get("healthy_stock_bonus", 0)
    )
    return adjusted

## 5. System prompt + action parser

In [ ]:
SYSTEM_PROMPT = """You are an AI supply chain manager for a 3-warehouse network (Seattle, Chicago, NYC).

Each step you choose ONE shipping action. Goal: maximize profit while keeping carbon emissions low.

Reply with exactly one line of valid JSON matching this schema:
{\"ship_amount\": <float>=0>, \"origin_city\": <\"Seattle\"|\"Chicago\"|\"NYC\">, \"destination_city\": <\"Seattle\"|\"Chicago\"|\"NYC\">, \"speed_mode\": <\"Air\"|\"Rail\">}

Rules:
- Rail: cheap, low-carbon, slow (3 steps).
- Air: fast (1 step), expensive, high-carbon.
- ship_amount=0 is a valid no-op.
- weather_alert field warns about disrupted or contested routes (higher cost) — route around them.
- Demand can spike unexpectedly — watch for sudden jumps in current_demand.

Respond with ONLY the JSON object. No explanation, no markdown."""

def format_observation_prompt(obs: dict) -> str:
    return (
        f"Step {obs['step_number']}/{obs['total_steps']}\n"
        f"Inventory: {obs['current_inventory']}\n"
        f"Demand this step: {obs['current_demand']}\n"
        f"Pending shipments: {obs.get('pending_shipments', [])}\n"
        f"Carbon credits left: {obs['carbon_credit_balance']:.1f}\n"
        f"Cumulative profit: {obs.get('cumulative_profit', 0):.2f}\n"
        f"Weather/market alert: {obs.get('weather_alert') or 'clear'}\n"
        f"\nYour action (JSON only):"
    )

In [ ]:
import json
import re

SAFE_FALLBACK_ACTION = {
    "ship_amount": 0.0,
    "origin_city": "Seattle",
    "destination_city": "Chicago",
    "speed_mode": "Rail",
}

def parse_action(completion: str):
    m = re.search(r"\{[^{}]*\}", completion, re.DOTALL)
    if not m:
        return SAFE_FALLBACK_ACTION, False
    try:
        action = json.loads(m.group(0))
        required = {"ship_amount", "origin_city", "destination_city", "speed_mode"}
        if not required.issubset(action.keys()):
            return SAFE_FALLBACK_ACTION, False
        return action, True
    except (json.JSONDecodeError, ValueError):
        return SAFE_FALLBACK_ACTION, False

## 6. Episode rollout helper

Every episode uses its own session ID so parallel rollouts don't corrupt each other's state.

In [ ]:
def run_episode(generate_fn, task_id=None, max_steps=None, env_url=None, seed=None):
    """Run one full episode with a unique session ID for parallel safety."""
    task_id = task_id or TASK_ID
    max_steps = max_steps or MAX_ENV_STEPS
    env_url = env_url or ENV_URL
    rng = random.Random(seed)
    session_id = f"train-{uuid.uuid4().hex[:12]}"
    headers = {"X-Session-Id": session_id}
    
    r = requests.post(f"{env_url}/reset", json={"task_id": task_id}, headers=headers, timeout=30)
    r.raise_for_status()
    obs = r.json()
    
    trajectory = []
    total_reward = total_profit = total_carbon = total_demand = total_fulfilled = 0.0
    valid_count = 0
    UNIT_PRICE = 10.0  # matches SELL_PRICE in env.py
    
    for step_idx in range(max_steps):
        wrapped_obs, shock_state = apply_world_wrapper(obs, rng)
        user_prompt = format_observation_prompt(wrapped_obs)
        full_prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{user_prompt}\n<|assistant|>\n"
        completion = generate_fn(full_prompt)
        action, was_valid = parse_action(completion)
        if was_valid:
            valid_count += 1
        
        r = requests.post(f"{env_url}/step", json=action, headers=headers, timeout=30)
        r.raise_for_status()
        step_resp = r.json()
        
        raw_reward = step_resp.get("reward", {})
        adjusted_reward = apply_reward_wrapper(raw_reward, action, shock_state)
        step_reward = adjusted_reward.get("total", 0.0)
        
        total_reward += step_reward
        total_profit += (adjusted_reward.get("sales_revenue", 0) - adjusted_reward.get("shipping_cost", 0))
        total_carbon += adjusted_reward.get("carbon_penalty", 0)
        step_demand = sum(wrapped_obs["current_demand"].values())
        total_demand += step_demand
        total_fulfilled += adjusted_reward.get("sales_revenue", 0) / UNIT_PRICE
        
        obs = step_resp.get("observation", obs)
        trajectory.append({
            "step": step_idx, "action": action, "valid": was_valid,
            "reward": step_reward,
            "profit": adjusted_reward.get("sales_revenue", 0) - adjusted_reward.get("shipping_cost", 0),
            "carbon": adjusted_reward.get("carbon_penalty", 0),
            "shocked": shock_state["demand_shocked"],
            "contested": shock_state["contested_route"] is not None,
        })
        
        if step_resp.get("done"):
            break
    
    try:
        g = requests.post(f"{env_url}/grader", json={"task_id": task_id}, headers=headers, timeout=30)
        grader_score = g.json().get("score", 0.0)
    except Exception:
        grader_score = 0.0
    
    return {
        "total_reward": total_reward,
        "profit": total_profit,
        "carbon_used": total_carbon,
        "carbon_efficiency": total_profit / max(1.0, total_carbon),
        "delivery_success_rate": total_fulfilled / max(1.0, total_demand),
        "valid_action_rate": valid_count / max(1, len(trajectory)),
        "grader_score": grader_score,
        "trajectory": trajectory,
    }

## 7. Load model with Unsloth + LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch
import numpy as np

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded.")

In [ ]:
# CRITICAL: no FastLanguageModel.for_inference() — that corrupts GRPO's gradient graph.
# torch.inference_mode() is stricter than no_grad() and required for mid-training generation.
def eager_generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

## 8. Baseline eval (BEFORE training)

In [ ]:
print("Running 10 BASELINE episodes...")
baseline_results = [run_episode(eager_generate, seed=i) for i in range(10)]

def summarize(results, label):
    print(f"\n=== {label} ===")
    for key in ["total_reward", "profit", "carbon_used", "carbon_efficiency",
                "delivery_success_rate", "valid_action_rate", "grader_score"]:
        vals = [r[key] for r in results]
        print(f"  {key:25s}: {np.mean(vals):+8.3f} \u00b1 {np.std(vals):.3f}")

summarize(baseline_results, "BEFORE TRAINING")

## 9. GRPO training

In [ ]:
class RunningStats:
    def __init__(self, eps=1e-6):
        self.n = 0
        self.mean = 0.0
        self.m2 = 0.0
        self.eps = eps
    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        self.m2 += delta * (x - self.mean)
    @property
    def std(self):
        return max(self.eps, (self.m2 / max(1, self.n - 1)) ** 0.5)

reward_stats = RunningStats()

training_log = {
    "grpo_step": [], "mean_reward": [], "mean_profit": [], "mean_carbon": [],
    "carbon_efficiency": [], "delivery_rate": [], "valid_rate": [], "grader_score": [],
}
global_step_counter = {"n": 0}

def env_reward_func(prompts, completions, **kwargs):
    """Standard GRPO reward function — runs one episode per completion, returns normalized reward."""
    rewards = []
    batch_metrics = {"reward": [], "profit": [], "carbon": [],
                     "carbon_eff": [], "delivery": [], "valid": [], "grader": []}
    
    for completion in completions:
        result = run_episode(eager_generate)
        raw_reward = result["total_reward"]
        reward_stats.update(raw_reward)
        normalized_reward = (raw_reward - reward_stats.mean) / reward_stats.std
        rewards.append(float(normalized_reward))
        batch_metrics["reward"].append(raw_reward)
        batch_metrics["profit"].append(result["profit"])
        batch_metrics["carbon"].append(result["carbon_used"])
        batch_metrics["carbon_eff"].append(result["carbon_efficiency"])
        batch_metrics["delivery"].append(result["delivery_success_rate"])
        batch_metrics["valid"].append(result["valid_action_rate"])
        batch_metrics["grader"].append(result["grader_score"])
    
    global_step_counter["n"] += 1
    training_log["grpo_step"].append(global_step_counter["n"])
    training_log["mean_reward"].append(np.mean(batch_metrics["reward"]))
    training_log["mean_profit"].append(np.mean(batch_metrics["profit"]))
    training_log["mean_carbon"].append(np.mean(batch_metrics["carbon"]))
    training_log["carbon_efficiency"].append(np.mean(batch_metrics["carbon_eff"]))
    training_log["delivery_rate"].append(np.mean(batch_metrics["delivery"]))
    training_log["valid_rate"].append(np.mean(batch_metrics["valid"]))
    training_log["grader_score"].append(np.mean(batch_metrics["grader"]))
    
    if global_step_counter["n"] % 5 == 0:
        print(f"[step {global_step_counter['n']}] "
              f"reward={np.mean(batch_metrics['reward']):.2f} "
              f"grader={np.mean(batch_metrics['grader']):.3f} "
              f"delivery={np.mean(batch_metrics['delivery']):.2%}")
    
    return rewards

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

train_dataset = Dataset.from_list([
    {"prompt": f"task:{TASK_ID}"} for _ in range(MAX_STEPS * GRAD_ACCUM_STEPS)
])

training_args = GRPOConfig(
    output_dir="./eco-logistics-grpo-v3",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=1024,
    max_steps=MAX_STEPS,
    logging_steps=1,
    save_steps=25,                                  # checkpoint every 25 steps
    bf16=USE_BF16,
    fp16=not USE_BF16,                              # auto-picks T4-safe fp16
    use_vllm=False,
    report_to="none",                               # no trackio (dependency conflict)
    run_name=f"eco-logistics-v3-{TASK_ID}",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=env_reward_func,
    processing_class=tokenizer,
)

trainer.train()

## 10. Auto-save plots & logs (downloads before Colab can disconnect)

**Run this cell immediately after training completes** — it saves artifacts to disk AND triggers browser downloads so you keep the files even if the runtime dies afterward.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Save CSV first (fastest, smallest)
log_df = pd.DataFrame(training_log)
csv_path = f"training_log_{TASK_ID}.csv"
log_df.to_csv(csv_path, index=False)
print(f"Saved {csv_path}")

# Generate plot
WINDOW = 5
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(log_df["grpo_step"], log_df["mean_reward"], alpha=0.3, label="per step")
if len(log_df) >= WINDOW:
    axes[0].plot(log_df["grpo_step"], log_df["mean_reward"].rolling(WINDOW).mean(),
                 linewidth=2, label=f"{WINDOW}-step rolling mean")
axes[0].set_xlabel("GRPO step")
axes[0].set_ylabel("Mean episode reward")
axes[0].set_title(f"Training reward — {TASK_ID}")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(log_df["grpo_step"], log_df["grader_score"], alpha=0.3, label="per step")
if len(log_df) >= WINDOW:
    axes[1].plot(log_df["grpo_step"], log_df["grader_score"].rolling(WINDOW).mean(),
                 linewidth=2, color="#2a9d8f", label=f"{WINDOW}-step rolling mean")
axes[1].set_xlabel("GRPO step")
axes[1].set_ylabel("Grader score (0-1)")
axes[1].set_title("Task completion over training")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
png_path = f"training_curves_{TASK_ID}.png"
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {png_path}")

# Trigger immediate browser download — survives Colab disconnects
try:
    from google.colab import files
    files.download(png_path)
    files.download(csv_path)
    print("Downloads triggered — check your browser")
except ImportError:
    print("Not on Colab — files saved locally only")

## 11. Post-training eval (AFTER)

In [ ]:
print("Running 10 AFTER episodes...")
trained_results = [run_episode(eager_generate, seed=i+1000) for i in range(10)]
summarize(baseline_results, "BEFORE TRAINING")
summarize(trained_results, "AFTER TRAINING")

print("\n=== DELTAS ===")
for key in ["total_reward", "profit", "carbon_used",
            "carbon_efficiency", "delivery_success_rate", "grader_score"]:
    before = np.mean([r[key] for r in baseline_results])
    after = np.mean([r[key] for r in trained_results])
    print(f"  {key:25s}: {before:+8.3f} \u2192 {after:+8.3f}  (\u0394 {after-before:+.3f})")

## 12. Trajectory dumps (for the pitch)

In [ ]:
def dump_trajectory(result, path):
    with open(path, "w") as f:
        f.write(f"Task: {TASK_ID}\n")
        f.write(f"Total reward: {result['total_reward']:.2f}\n")
        f.write(f"Profit: {result['profit']:.2f}\n")
        f.write(f"Carbon used: {result['carbon_used']:.2f}\n")
        f.write(f"Delivery rate: {result['delivery_success_rate']:.1%}\n")
        f.write(f"Grader score: {result['grader_score']:.3f}\n\n")
        for t in result["trajectory"]:
            flags = []
            if t["shocked"]: flags.append("DEMAND_SHOCK")
            if t["contested"]: flags.append("COMPETITOR_BID")
            flag_str = f" [{', '.join(flags)}]" if flags else ""
            f.write(f"Step {t['step']}{flag_str}: {t['action']}\n")
            f.write(f"  reward={t['reward']:+.2f}  profit={t['profit']:+.2f}  carbon={t['carbon']:.2f}  valid={t['valid']}\n")
    print(f"Wrote {path}")

def median_idx(results, key="total_reward"):
    vals = [r[key] for r in results]
    return sorted(range(len(vals)), key=lambda i: vals[i])[len(vals) // 2]

before_path = f"trajectory_before_{TASK_ID}.txt"
after_path = f"trajectory_after_{TASK_ID}.txt"
dump_trajectory(baseline_results[median_idx(baseline_results)], before_path)
dump_trajectory(trained_results[median_idx(trained_results)], after_path)

try:
    from google.colab import files
    files.download(before_path)
    files.download(after_path)
    print("Downloaded both trajectory files")
except ImportError:
    print("Not on Colab")

## 13. Save LoRA adapter (optional — for submission)

In [ ]:
adapter_path = f"eco-logistics-lora-v3-{TASK_ID}"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved to {adapter_path}")

# Uncomment + add your HF username to push:
# model.push_to_hub(f"YOUR_USERNAME/{adapter_path}")
# tokenizer.push_to_hub(f"YOUR_USERNAME/{adapter_path}")

## Appendix — troubleshooting

**`ValueError: bf16/gpu not supported`** → Set `USE_BF16 = False` in cell 5. T4 doesn't support bf16.

**`RuntimeError: inplace operation modified gradient`** → You edited `eager_generate` to add `FastLanguageModel.for_inference(model)`. Don't. The version in this notebook is intentional.

**`ModuleNotFoundError: trl`** → Colab runtime disconnected. Re-run cell 2 (installs) after `Runtime → Restart session`.

**`ValueError: too many values to unpack (expected 3)`** → Your env.py returns a different tuple shape. Check the smoke-test cell's `step_resp` keys.

**Colab disconnects mid-training** → The auto-download cell (10) handles this as long as it ran at least once. If training didn't even reach cell 10, use checkpoint-25 or -50 via `save_steps=25`.

**Grader doesn't improve after 100 steps** → Try `net_zero_profit` task (harder, more room to improve) or lower `LEARNING_RATE` to 1e-6.